# BƯỚC 3: Trích xuất Đặc trưng Hình ảnh
## Feature Engineering - Crop & Extract Features

Cầu nối giữa Deep Learning (YOLO) và Machine Learning
- Sử dụng YOLO model để crop cell từ ảnh gốc
- Trích xuất đặc trưng hình thái, màu sắc, kết cấu
- Lưu vào features.csv để train ML models

## 1. Setup & Import Libraries

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path.cwd()
print(f"Working directory: {BASE_DIR}")
print("✓ All libraries imported!")

## 2. Load YOLO Model

In [ ]:
# Load best.pt model
model_path = BASE_DIR / 'models/yolo/best.pt'

if model_path.exists():
    yolo_model = YOLO(str(model_path))
    print(f"✓ YOLO model loaded from: {model_path}")
else:
    print(f"⚠️ Model not found at {model_path}")
    print("   → Copy best.pt từ Kaggle training vào models/yolo/best.pt")

## 3. Định nghĩa Functions để Trích xuất Đặc trưng

In [ ]:
class CellFeatureExtractor:
    """Trích xuất đặc trưng từ ảnh cell"""
    
    @staticmethod
    def extract_morphology_features(binary_img):
        """Trích xuất đặc trưng hình thái"""
        features = {}
        
        # Area
        area = cv2.countNonZero(binary_img)
        features['area'] = area
        
        # Perimeter
        contours, _ = cv2.findContours(binary_img, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
        if len(contours) > 0:
            cnt = max(contours, key=cv2.contourArea)
            perimeter = cv2.arcLength(cnt, True)
            features['perimeter'] = perimeter
            
            # Circularity (4 * π * area / perimeter²)
            if perimeter > 0:
                circularity = 4 * np.pi * area / (perimeter ** 2)
                features['circularity'] = circularity
            
            # Eccentricity
            if len(cnt) >= 5:
                ellipse = cv2.fitEllipse(cnt)
                (x, y), (MA, ma), angle = ellipse
                if MA > 0:
                    eccentricity = np.sqrt(1 - (ma / MA) ** 2)
                    features['eccentricity'] = eccentricity
        
        return features
    
    @staticmethod
    def extract_color_features(img):
        """Trích xuất đặc trưng màu sắc"""
        features = {}
        
        # RGB means
        features['mean_B'] = np.mean(img[:,:,0])
        features['mean_G'] = np.mean(img[:,:,1])
        features['mean_R'] = np.mean(img[:,:,2])
        
        # RGB std
        features['std_B'] = np.std(img[:,:,0])
        features['std_G'] = np.std(img[:,:,1])
        features['std_R'] = np.std(img[:,:,2])
        
        # HSV conversion
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV).astype(float)
        features['mean_H'] = np.mean(hsv[:,:,0])
        features['mean_S'] = np.mean(hsv[:,:,1])
        features['mean_V'] = np.mean(hsv[:,:,2])
        
        return features
    
    @staticmethod
    def extract_texture_features(gray_img):
        """Trích xuất đặc trưng kết cấu"""
        features = {}
        
        # Edge detection
        edges = cv2.Canny(gray_img, 50, 150)
        features['edge_density'] = np.sum(edges > 0) / edges.size
        
        # Entropy (texture complexity)
        hist = cv2.calcHist([gray_img], [0], None, [256], [0, 256])
        hist = hist.flatten() / hist.sum()
        entropy = -np.sum(hist[hist > 0] * np.log2(hist[hist > 0]))
        features['entropy'] = entropy
        
        # Contrast
        contrast = np.std(gray_img)
        features['contrast'] = contrast
        
        return features
    
    @staticmethod
    def extract(img, label=None):
        """Trích xuất tất cả đặc trưng từ cell image"""
        features = {}
        
        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Thresholding
        _, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
        
        # Extract all features
        features.update(CellFeatureExtractor.extract_morphology_features(binary))
        features.update(CellFeatureExtractor.extract_color_features(img))
        features.update(CellFeatureExtractor.extract_texture_features(gray))
        
        if label is not None:
            features['label'] = label
        
        return features

print("✓ Feature Extractor class defined")

## 4. Crop Cell Images từ YOLO Detection

In [ ]:
print("\n" + "="*60)
print("🔪 CROPPING CELLS FROM DETECTION RESULTS")
print("="*60)

crops_output_dir = BASE_DIR / 'data/processed/crops'
crops_output_dir.mkdir(parents=True, exist_ok=True)

# Lấy ảnh từ data/raw/images
images_dir = BASE_DIR / 'data/raw/images'
image_files = sorted(list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png')))[:20]  # 20 ảnh đầu

print(f"\nProcessing {len(image_files)} images...\n")

crop_count = 0
for img_path in tqdm(image_files, desc="Cropping cells"):
    img = cv2.imread(str(img_path))
    if img is None:
        continue
    
    # Predict
    results = yolo_model.predict(str(img_path), conf=0.25, verbose=False)
    
    # Crop từng cell
    for i, box in enumerate(results[0].boxes):
        x1, y1, x2, y2 = box.xyxy[0].int().tolist()
        
        # Crop
        crop = img[y1:y2, x1:x2]
        
        if crop.size > 0:
            # Save crop
            crop_name = f"{img_path.stem}_cell_{i}.jpg"
            crop_path = crops_output_dir / crop_name
            cv2.imwrite(str(crop_path), crop)
            crop_count += 1

print(f"\n✓ Cropped {crop_count} cells")

## 5. Extract Features từ Crops

In [ ]:
print("\n" + "="*60)
print("🔬 EXTRACTING FEATURES FROM CROPS")
print("="*60)

extractor = CellFeatureExtractor()
features_list = []

crop_files = sorted(list(crops_output_dir.glob('*.jpg')))
print(f"\nExtracting features from {len(crop_files)} crops...\n")

for crop_path in tqdm(crop_files, desc="Extracting features"):
    crop = cv2.imread(str(crop_path))
    if crop is None:
        continue
    
    # Extract features
    features = extractor.extract(crop)
    features['image_name'] = crop_path.name
    
    features_list.append(features)

# Tạo DataFrame
df_features = pd.DataFrame(features_list)
print(f"\n✓ Extracted features shape: {df_features.shape}")
print(f"\nFeature columns ({len(df_features.columns)}):")
for col in df_features.columns:
    print(f"  - {col}")

## 6. Data Cleaning & Preprocessing

In [ ]:
print("\n" + "="*60)
print("🧹 DATA CLEANING")
print("="*60)

# Check missing values
missing = df_features.isnull().sum()
if missing.sum() > 0:
    print(f"\nMissing values found:")
    print(missing[missing > 0])
    
    # Fill with mean
    df_features = df_features.fillna(df_features.mean())
    print("✓ Filled with mean values")
else:
    print("✓ No missing values")

# Remove duplicates
df_features = df_features.drop_duplicates()
print(f"✓ After removing duplicates: {len(df_features)} samples")

# Statistics
print(f"\nFeature statistics:")
print(df_features.describe())

## 7. Lưu Features vào CSV

In [ ]:
print("\n" + "="*60)
print("💾 SAVING FEATURES")
print("="*60)

features_output = BASE_DIR / 'data/processed/features/train_features.csv'
features_output.parent.mkdir(parents=True, exist_ok=True)

df_features.to_csv(features_output, index=False)
print(f"\n✓ Features saved to: {features_output}")
print(f"  Shape: {df_features.shape}")
print(f"  File size: {features_output.stat().st_size / (1024*1024):.2f} MB")

print(f"\nFirst few rows:")
print(df_features.head())

## 8. Visualization

In [ ]:
# Plot feature distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')

feature_cols = ['area', 'perimeter', 'circularity', 'mean_R', 'mean_G', 'mean_B']
for idx, col in enumerate(feature_cols):
    ax = axes[idx // 3, idx % 3]
    if col in df_features.columns:
        ax.hist(df_features[col], bins=20, alpha=0.7, color='blue')
        ax.set_xlabel(col)
        ax.set_ylabel('Frequency')
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Tóm tắt Bước 3

In [ ]:
print("\n" + "="*60)
print("✅ BƯỚC 3 HOÀN THÀNH - FEATURE EXTRACTION")
print("="*60)
print(f"""
✓ Cropped {crop_count} cells from images
✓ Extracted {len(df_features)} feature sets
✓ Features saved to: data/processed/features/train_features.csv

📊 Feature Summary:
   - Total samples: {len(df_features)}
   - Total features: {len(df_features.columns)}
   - Feature types: Morphology, Color, Texture

📥 BƯỚC TIẾP THEO:
   Bước 4: Train ML Classification Models (train_ml.ipynb)
   
💾 Output file:
   - data/processed/features/train_features.csv
   
📝 Đặc trưng được trích xuất:
   - Morphology: area, perimeter, circularity, eccentricity
   - Color: mean/std RGB, mean HSV
   - Texture: edge_density, entropy, contrast
""")